# 01 — Data Quality and SQL Loading

This notebook prepares the raw technology stock price dataset for the project analytics workflow. It:

- loads the raw CSV through the project loader in `src.data_loader`,
- validates basic OHLCV data quality constraints,
- creates the local DuckDB database through `src.sql_utils.initialize_database`, and
- inspects the SQL `data_quality_summary` view that downstream feature-engineering notebooks can build on.

The dataset contains historical daily open, high, low, close, adjusted close, volume, and split coefficient fields for large-cap technology equities. The checks below are descriptive quality controls only; they do not imply predictive power or investment suitability.


## 2. Imports and path setup

The project modules live in `src/`. The setup below makes the notebook runnable from either the repository root or the `notebooks/` directory.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Resolve repository root whether the notebook is launched from the root or notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DUCKDB_PATH, FIGURES_DIR, PROCESSED_DATA_DIR, RAW_CSV_PATH
from src.data_loader import PRICE_COLUMNS, load_raw_stock_data
from src.sql_utils import initialize_database, query_to_df

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)
plt.style.use("seaborn-v0_8-whitegrid")

# Prefer the configured data/raw location. Fall back to the repository-root CSV
# so this notebook remains runnable in lightweight clones of the project.
raw_csv_candidates = [RAW_CSV_PATH, PROJECT_ROOT / "tech_stocks.csv"]
RAW_CSV_FOR_NOTEBOOK = next((path for path in raw_csv_candidates if path.exists()), RAW_CSV_PATH)

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw CSV path: {RAW_CSV_FOR_NOTEBOOK}")
print(f"DuckDB path: {DUCKDB_PATH}")


## 3. Load raw CSV using `src.data_loader.load_raw_stock_data`

The loader parses dates, validates required columns, converts numeric fields, checks positive price/volume values, and returns records sorted by `symbol` and `date`.


In [ ]:
prices = load_raw_stock_data(RAW_CSV_FOR_NOTEBOOK)
prices["date"] = pd.to_datetime(prices["date"])

print(f"Loaded {len(prices):,} rows across {prices['symbol'].nunique():,} symbols.")


## 4. Dataset shape, columns, dtypes, and first rows

These quick structural checks confirm the table dimensions and the field types that downstream analysis will use.


In [ ]:
shape_summary = pd.DataFrame(
    {"metric": ["rows", "columns"], "value": [prices.shape[0], prices.shape[1]]}
)
columns_summary = pd.DataFrame(
    {"position": range(1, len(prices.columns) + 1), "column": prices.columns}
)
dtypes_summary = prices.dtypes.astype(str).reset_index()
dtypes_summary.columns = ["column", "dtype"]

display(shape_summary)
display(columns_summary)
display(dtypes_summary)
display(prices.head())


## 5. Date range by symbol

Symbols can have different historical coverage depending on listing history and data availability. The row counts below should therefore be interpreted by symbol rather than assuming a common start date.


In [ ]:
date_range_by_symbol = (
    prices.groupby("symbol", as_index=False)
    .agg(
        min_date=("date", "min"),
        max_date=("date", "max"),
        row_count=("date", "size"),
        distinct_date_count=("date", "nunique"),
    )
    .sort_values("symbol")
)

display(date_range_by_symbol)

coverage_plot = date_range_by_symbol.set_index("symbol")["row_count"].plot(
    kind="bar",
    figsize=(8, 4),
    title="Rows by Symbol",
    ylabel="Trading-day rows",
)
coverage_plot.set_xlabel("Symbol")
plt.tight_layout()
plt.show()


## 6. Missing value analysis

The loader validates required numeric fields, but this table is still useful documentation for GitHub readers and future data refreshes.


In [ ]:
missing_summary = (
    prices.isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
missing_summary["missing_pct"] = missing_summary["missing_count"] / len(prices)

display(missing_summary)

ax = missing_summary.plot(
    x="column",
    y="missing_count",
    kind="bar",
    legend=False,
    figsize=(9, 4),
    title="Missing Values by Column",
)
ax.set_xlabel("Column")
ax.set_ylabel("Missing count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 7. Duplicate symbol-date check

Daily OHLCV analysis expects at most one record per `symbol` and `date`. Duplicate rows would require either deduplication or a documented aggregation rule before returns are calculated.


In [ ]:
duplicate_mask = prices.duplicated(subset=["symbol", "date"], keep=False)
duplicate_symbol_date_count = int(duplicate_mask.sum())

duplicate_summary = (
    prices.loc[duplicate_mask]
    .groupby(["symbol", "date"], as_index=False)
    .size()
    .rename(columns={"size": "duplicate_records"})
    .sort_values(["symbol", "date"])
)

print(f"Duplicate symbol-date rows: {duplicate_symbol_date_count:,}")
display(duplicate_summary.head(20))


## 8. Price sanity checks

The following checks cover basic OHLC and adjusted-close consistency:

- non-positive `open`, `high`, `low`, `close`, or `close_adjusted`,
- `high >= low`, and
- `close` within the intraday high-low range for rows where high/low are otherwise valid.


In [ ]:
non_positive_prices = (
    (prices[list(PRICE_COLUMNS)] <= 0)
    .sum()
    .rename("non_positive_count")
    .reset_index()
    .rename(columns={"index": "price_column"})
)

invalid_high_low = prices.loc[prices["high"] < prices["low"], ["symbol", "date", "high", "low"]]
valid_high_low = prices["high"] >= prices["low"]
close_outside_range = prices.loc[
    valid_high_low & ~prices["close"].between(prices["low"], prices["high"]),
    ["symbol", "date", "low", "close", "high"],
]

price_check_summary = pd.DataFrame(
    {
        "check": [
            "non_positive_price_cells",
            "rows_with_high_below_low",
            "rows_with_close_outside_high_low",
        ],
        "count": [
            int(non_positive_prices["non_positive_count"].sum()),
            len(invalid_high_low),
            len(close_outside_range),
        ],
    }
)

display(non_positive_prices)
display(price_check_summary)
display(invalid_high_low.head(10))
display(close_outside_range.head(10))


## 9. Volume sanity checks

Volume should be positive for all rows loaded by the project loader. The distribution below helps identify unusually small or large trading-volume observations for review.


In [ ]:
volume_summary = prices.groupby("symbol")["volume"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
non_positive_volume_count = int((prices["volume"] <= 0).sum())

print(f"Rows with non-positive volume: {non_positive_volume_count:,}")
display(volume_summary)

ax = prices.boxplot(column="volume", by="symbol", figsize=(9, 4), rot=45)
ax.set_title("Volume Distribution by Symbol")
ax.set_xlabel("Symbol")
ax.set_ylabel("Volume")
plt.suptitle("")
plt.tight_layout()
plt.show()


## 10. Split coefficient summary

Split coefficients document corporate actions that affect raw prices. Adjusted close is generally preferred for return calculations because it accounts for splits and other adjustments represented in the adjusted price series.


In [ ]:
split_summary = prices.groupby("symbol")["split_coefficient"].describe()
split_events = prices.loc[
    prices["split_coefficient"].ne(1),
    ["symbol", "date", "split_coefficient"],
].sort_values(["symbol", "date"])

print(f"Rows with split coefficient different from 1: {len(split_events):,}")
display(split_summary)
display(split_events.head(20))


## 11. Create DuckDB database using `src.sql_utils.initialize_database`

`initialize_database` creates or replaces the `raw_prices` table and runs the clean-price SQL script that defines the analysis-ready `clean_prices` table and the `data_quality_summary` view.


In [ ]:
connection = initialize_database(raw_csv_path=RAW_CSV_FOR_NOTEBOOK, db_path=DUCKDB_PATH)

loaded_tables = query_to_df(
    connection,
    '''
    SELECT table_name, table_type
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
    ''',
)

display(loaded_tables)


## 12. Query `data_quality_summary` from SQL

This view provides a compact SQL-side coverage and range check by symbol after the raw data has been loaded and cleaned.


In [ ]:
sql_quality_summary = query_to_df(
    connection,
    '''
    SELECT
        symbol,
        min_date,
        max_date,
        row_count,
        distinct_date_count,
        average_volume,
        min_adj_close,
        max_adj_close
    FROM data_quality_summary
    ORDER BY symbol
    ''',
)

display(sql_quality_summary)


## 13. Save basic summary tables and figures

The notebook saves lightweight CSV summaries and a row-coverage plot so later notebooks or reports can reuse the quality-control outputs without recomputing them manually.


In [ ]:
date_range_path = PROCESSED_DATA_DIR / "data_quality_date_range_by_symbol.csv"
missing_summary_path = PROCESSED_DATA_DIR / "data_quality_missing_values.csv"
sql_quality_path = PROCESSED_DATA_DIR / "data_quality_sql_summary.csv"
coverage_figure_path = FIGURES_DIR / "data_quality_rows_by_symbol.png"

# Save tables.
date_range_by_symbol.to_csv(date_range_path, index=False)
missing_summary.to_csv(missing_summary_path, index=False)
sql_quality_summary.to_csv(sql_quality_path, index=False)

# Save a simple, report-friendly coverage chart.
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(date_range_by_symbol["symbol"], date_range_by_symbol["row_count"])
ax.set_title("Rows by Symbol")
ax.set_xlabel("Symbol")
ax.set_ylabel("Trading-day rows")
plt.tight_layout()
fig.savefig(coverage_figure_path, dpi=150, bbox_inches="tight")
plt.show()

saved_outputs = pd.DataFrame(
    {
        "artifact": [
            "date range summary",
            "missing value summary",
            "SQL quality summary",
            "row coverage figure",
        ],
        "path": [
            date_range_path,
            missing_summary_path,
            sql_quality_path,
            coverage_figure_path,
        ],
    }
)

display(saved_outputs)


## 14. Conclusion

The raw dataset contains historical daily OHLCV records for technology stocks. The quality checks above confirm the basic structure needed for financial feature engineering while also documenting that symbols can have different historical date ranges. For return calculations, downstream notebooks should use `close_adjusted` / `adj_close` so split-adjusted price history is reflected consistently. The DuckDB pipeline is now initialized and ready for SQL-based feature engineering in the next stage of the project.


In [ ]:
connection.close()
print("DuckDB connection closed. Data-quality notebook complete.")
